In [ ]:
# Parametric PINNs for Navier-Stokes ( Backward Facing Step Flow )
# Aditya Jangir ( IIT Delhi )
# This Jupyter Notebook contains code for physics-informed neural networks (PINNs).
# We refer it as Data Free PINNs as it does not need any CFD data in the trainning process.
# It solves the incompressible Navier–Stokes equations for a wide range of Reynolds numbers.
# Instead of training the model for one specific flow case,
# the network is trained to learn a general solution that works across many flow conditions.
# The results generated using this code are reported in the paper
# “Sparse-Supervised Hybrid Parameterized Physics-Informed Neural Networks for Incompressible Flows Across Reynolds Numbers”,
# Which is available on arXiv: arXiv:2602.04670 and is being prepared for submission to Physics of Fluids.

In [ ]:
# import relevent libraries and packages
import numpy as np
import time
import os
import tensorflow as tf
import scipy.optimize as spo
import matplotlib.pyplot as plt
from matplotlib import cm
import subprocess

In [ ]:
# Problem / numerical params
tf.random.set_seed(42); np.random.seed(42)

# Reynolds number traininng range
Re_min, Re_max = 5e1, 3e2

# Geometry: backward-facing step
L_in  = 2.0      # inlet length
L_out = 9.0      # outlet length
h     = 0.5      # inlet height
H     = 1.0      # expanded channel height

# Domain bounds (global box that contains the step geometry)
x_min, x_max = -L_in, L_out
y_min, y_max = 0.0, H

# Neural network
D_default = 10
N_default = 80

# Collocation points
Nx_cells = 100
Ny_cells = 60

# Boundary sampling
N_b_inlet   = 80
N_b_outlet  = 80
N_b_walls   = 300

In [ ]:
# Ensure GPU usage
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
        print("GPU is available and will be used.")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU available. Running on CPU.")

In [ ]:
# Neural Network Architecture for Data Free PINNs
class PurePINN(tf.keras.Model):
    def __init__(self,
                 num_hidden_layers=D_default,
                 num_neurons_per_layer=N_default,
                 include_pressure=True,
                 activation='tanh',
                 kernel_regularizer=tf.keras.regularizers.L2(1e-8),
                x_mean=(x_min + x_max)/2,
                x_scale=(x_max - x_min)/2,
                y_mean=(y_min + y_max)/2,
                y_scale=(y_max - y_min)/2,
                re_mean=np.log(500),
                re_scale=1.0,
                 use_log_re=True,
                 **kwargs):
        super().__init__(**kwargs)

        self.num_hidden_layers = num_hidden_layers
        self.num_neurons_per_layer = num_neurons_per_layer
        self.include_pressure = include_pressure
        self.use_log_re = use_log_re

        # Normalization constants
        self.x_mean = tf.constant(x_mean, dtype=tf.float32)
        self.x_scale = tf.constant(x_scale, dtype=tf.float32)
        self.y_mean = tf.constant(y_mean, dtype=tf.float32)
        self.y_scale = tf.constant(y_scale, dtype=tf.float32)
        self.re_mean = tf.constant(re_mean, dtype=tf.float32)
        self.re_scale = tf.constant(re_scale, dtype=tf.float32)

        # Input normalization layer
        self.input_norm = tf.keras.layers.Lambda(self._normalize_inputs, name='input_norm')

        # Hidden layers
        hidden = []
        for i in range(num_hidden_layers):
            hidden.append(tf.keras.layers.Dense(num_neurons_per_layer,
                                                activation=activation,
                                                kernel_initializer='glorot_normal',
                                                kernel_regularizer=kernel_regularizer,
                                                name=f'dense_{i}'))
            hidden.append(tf.keras.layers.LayerNormalization(name=f'ln_{i}'))
        self.hidden_seq = tf.keras.Sequential(hidden, name='hidden_seq')

        # Output layers
        self.u_out = tf.keras.layers.Dense(1, activation=None, name='u_out')
        self.v_out = tf.keras.layers.Dense(1, activation=None, name='v_out')
        if self.include_pressure:
            self.p_out = tf.keras.layers.Dense(1, activation=None, name='p_out')
        else:
            self.p_out = None

    def _normalize_inputs(self, X):
        x = X[:, 0:1]
        y = X[:, 1:2]
        re = X[:, 2:3]

        x_n = (x - self.x_mean) / (self.x_scale + 1e-8)
        y_n = (y - self.y_mean) / (self.y_scale + 1e-8)
        if self.use_log_re:
            re_n = (tf.math.log(tf.maximum(re, 1e-8)) - self.re_mean) / (self.re_scale + 1e-8)
        else:
            re_n = (re - tf.exp(self.re_mean)) / (self.re_scale + 1e-8)

        return tf.concat([x_n, y_n, re_n], axis=1)

    def call(self, inputs, training=False):
        xn = self.input_norm(inputs)
        h = self.hidden_seq(xn, training=training)
        u = self.u_out(h)
        v = self.v_out(h)
        if self.include_pressure:
            p = self.p_out(h)
        else:
            p = tf.zeros_like(u)
        return u, v, p


In [ ]:
# Domain sampling for backward-facing step flow (steady, incompressible, low Re)
# Collocation points sampling (with refinement)

def stratified_interior_samples(nx_cells, ny_cells,
                               x0=x_min, x1=x_max,
                               y0=y_min, y1=y_max):

    xs, ys = [], []
    dx = (x1 - x0) / nx_cells
    dy = (y1 - y0) / ny_cells

    # Base stratified sampling
    for i in range(nx_cells):
        for j in range(ny_cells):
            x = x0 + i * dx + np.random.rand() * dx
            y = y0 + j * dy + np.random.rand() * dy

            # Remove solid region (step block)
            if not (x < 0.0 and y < h):
                xs.append(x)
                ys.append(y)

    xs = np.array(xs).reshape(-1, 1)
    ys = np.array(ys).reshape(-1, 1)

    # Extra refinement near step region
    N_refine = 2000

    x_ref = np.random.rand(N_refine, 1) * 5.0
    y_ref = np.random.rand(N_refine, 1) * H

    mask = ~((x_ref < 0.0) & (y_ref < h))

    # Apply mask and FIX SHAPE
    x_ref = x_ref[mask].reshape(-1, 1)
    y_ref = y_ref[mask].reshape(-1, 1)

    # Safe concatenation
    xs = np.vstack([xs, x_ref])
    ys = np.vstack([ys, y_ref])

    return xs, ys

# Boundary points sampling
def boundary_samples():

    # Inlet (upper channel only)
    y_in = np.random.rand(N_b_inlet, 1) * (H - h) + h
    x_in = np.ones_like(y_in) * (-L_in)
    inlet = np.hstack([x_in, y_in, np.zeros_like(y_in)])

    # Outlet (full height)
    y_out = np.random.rand(N_b_outlet, 1) * H
    x_out = np.ones_like(y_out) * L_out
    outlet = np.hstack([x_out, y_out, np.zeros_like(y_out)])

    # Top wall (FULL domain: upstream + downstream)
    x_top = np.random.rand(N_b_walls, 1) * (L_out + L_in) - L_in
    y_top = np.ones_like(x_top) * H
    top = np.hstack([x_top, y_top, np.zeros_like(x_top)])

    # Bottom wall BEFORE step (y = h)
    x_bot1 = np.random.rand(N_b_walls, 1) * L_in - L_in
    y_bot1 = np.ones_like(x_bot1) * h
    bottom1 = np.hstack([x_bot1, y_bot1, np.zeros_like(x_bot1)])

    # Bottom wall AFTER step (y = 0)
    x_bot2 = np.random.rand(N_b_walls, 1) * L_out
    y_bot2 = np.zeros_like(x_bot2)
    bottom2 = np.hstack([x_bot2, y_bot2, np.zeros_like(x_bot2)])

    # Step vertical wall
    y_step = np.random.rand(N_b_walls, 1) * h
    x_step = np.zeros_like(y_step)
    step = np.hstack([x_step, y_step, np.zeros_like(y_step)])

    # Corner (important junction)
    corner = np.array([[0.0, h, 0.0]])

    return inlet, outlet, top, bottom1, bottom2, step, corner

# Reynolds number sampling

def sample_Re(n, log_uniform=True):
    if log_uniform:
        logmin = np.log(Re_min)
        logmax = np.log(Re_max)
        return np.exp(np.random.rand(n) * (logmax - logmin) + logmin).reshape(-1, 1)
    else:
        return (np.random.rand(n, 1) * (Re_max - Re_min) + Re_min)

In [ ]:
# PDE residuals definition
def pde_residuals(model, X_collocation, Re_vals):

    # Combine inputs
    X_input = tf.concat([X_collocation, Re_vals], axis=1)
    X_input = tf.cast(X_input, tf.float32)

    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch(X_input)

        with tf.GradientTape(persistent=True) as tape:
            tape.watch(X_input)

            u, v, p = model(X_input, training=True)

        # First derivatives 
        grads_u = tape.gradient(u, X_input)
        grads_v = tape.gradient(v, X_input)
        grads_p = tape.gradient(p, X_input)

        u_x, u_y = grads_u[:, 0:1], grads_u[:, 1:2]
        v_x, v_y = grads_v[:, 0:1], grads_v[:, 1:2]
        p_x, p_y = grads_p[:, 0:1], grads_p[:, 1:2]

    # Second derivatives
    u_xx = tape2.gradient(u_x, X_input)[:, 0:1]
    u_yy = tape2.gradient(u_y, X_input)[:, 1:2]
    v_xx = tape2.gradient(v_x, X_input)[:, 0:1]
    v_yy = tape2.gradient(v_y, X_input)[:, 1:2]

    # Continuity
    r1 = u_x + v_y

    # Reynolds number (treated as parameter)
    inv_Re = 1.0 / (Re_vals + 1e-12)

    # Momentum
    r2 = u * u_x + v * u_y + p_x - inv_Re * (u_xx + u_yy)
    r3 = u * v_x + v * v_y + p_y - inv_Re * (v_xx + v_yy)

    del tape, tape2

    return r1, r2, r3

In [ ]:
# Boundary loss definition
def boundary_losses(model,
                    inlet, outlet, top, bottom1, bottom2, step, corner,
                    Re_in, Re_out, Re_top, Re_bot1, Re_bot2, Re_step, Re_corner):
   
    # Inputs
    inlet_in  = tf.concat([inlet[:,0:1],  inlet[:,1:2],  Re_in], axis=1)
    outlet_in = tf.concat([outlet[:,0:1], outlet[:,1:2], Re_out], axis=1)
    top_in    = tf.concat([top[:,0:1],    top[:,1:2],    Re_top], axis=1)

    bot1_in   = tf.concat([bottom1[:,0:1], bottom1[:,1:2], Re_bot1], axis=1)
    bot2_in   = tf.concat([bottom2[:,0:1], bottom2[:,1:2], Re_bot2], axis=1)

    step_in   = tf.concat([step[:,0:1],   step[:,1:2],   Re_step], axis=1)

    # Predictions
    u_in, v_in, _ = model(inlet_in, training=True)
    _, _, p_out   = model(outlet_in, training=True)

    u_top, v_top, _ = model(top_in, training=True)
    u_bot1, v_bot1, _ = model(bot1_in, training=True)
    u_bot2, v_bot2, _ = model(bot2_in, training=True)
    u_step, v_step, _ = model(step_in, training=True)

    # Inlet (parabolic profile)
    y = inlet[:,1:2]
    u_target = 4.0 * (y - h) * (H - y) / (H - h)**2

    bc_inlet = tf.reduce_mean((u_in - u_target)**2 + v_in**2)

    # Outlet (pressure)
    bc_outlet = tf.reduce_mean(p_out**2)

    # No-slip walls (ALL walls)
    bc_top   = tf.reduce_mean(u_top**2 + v_top**2)
    bc_bot1  = tf.reduce_mean(u_bot1**2 + v_bot1**2)
    bc_bot2  = tf.reduce_mean(u_bot2**2 + v_bot2**2)
    bc_step  = tf.reduce_mean(u_step**2 + v_step**2)

    # Total losses
    Lbc_v = bc_inlet + bc_top + bc_bot1 + bc_bot2 + bc_step
    Lbc_p = bc_outlet

    return Lbc_v, Lbc_p

In [ ]:
# Loss Function
@tf.function
def training_step(model, optimizer,
                  X_interior, Re_interior,
                  inlet, outlet, top,
                  bottom1, bottom2, step, corner,
                  Re_in, Re_out, Re_top,
                  Re_bot1, Re_bot2, Re_step, Re_corner):

    with tf.GradientTape() as tape:

        # PDE residuals
        r1, r2, r3 = pde_residuals(model, X_interior, Re_interior)

        Le1 = tf.reduce_mean(tf.square(r1))
        Le2 = tf.reduce_mean(tf.square(r2))
        Le3 = tf.reduce_mean(tf.square(r3))

        # Boundary losses
        Lbc_v, Lbc_p = boundary_losses(
            model,
            inlet, outlet, top,
            bottom1, bottom2, step, corner,
            Re_in, Re_out, Re_top,
            Re_bot1, Re_bot2, Re_step, Re_corner
        )

        # Total loss
        loss_total = Le1 + Le2 + Le3 + Lbc_v + Lbc_p

    # Gradients
    grads = tape.gradient(loss_total, model.trainable_variables)
    grads = [tf.zeros_like(v) if g is None else g
             for g, v in zip(grads, model.trainable_variables)]

    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    # Gradient norm (monitoring)
    grad_norm = tf.linalg.global_norm(grads)

    return loss_total, Le1, Le2, Le3, Lbc_v, Lbc_p, grad_norm

In [ ]:

# Interior sampling
xs, ys = stratified_interior_samples(Nx_cells, Ny_cells)
N_interior = xs.shape[0]
Re_interior = sample_Re(N_interior, log_uniform=True)

# Boundary sampling
inlet_np, outlet_np, top_np, bottom1_np, bottom2_np, step_np, corner_np = boundary_samples()

# Reynolds sampling for each boundary
Re_in_np      = sample_Re(inlet_np.shape[0])
Re_out_np     = sample_Re(outlet_np.shape[0])
Re_top_np     = sample_Re(top_np.shape[0])
Re_bot1_np    = sample_Re(bottom1_np.shape[0])
Re_bot2_np    = sample_Re(bottom2_np.shape[0])
Re_step_np    = sample_Re(step_np.shape[0])
Re_corner_np  = sample_Re(corner_np.shape[0])

# Convert to tensors
X_interior_tf  = tf.convert_to_tensor(np.hstack([xs, ys]).astype(np.float32))
Re_interior_tf = tf.convert_to_tensor(Re_interior.astype(np.float32))

inlet_tf   = tf.convert_to_tensor(inlet_np[:, 0:2].astype(np.float32))
outlet_tf  = tf.convert_to_tensor(outlet_np[:, 0:2].astype(np.float32))
top_tf     = tf.convert_to_tensor(top_np[:, 0:2].astype(np.float32))

bottom1_tf = tf.convert_to_tensor(bottom1_np[:, 0:2].astype(np.float32))
bottom2_tf = tf.convert_to_tensor(bottom2_np[:, 0:2].astype(np.float32))

step_tf    = tf.convert_to_tensor(step_np[:, 0:2].astype(np.float32))
corner_tf  = tf.convert_to_tensor(corner_np[:, 0:2].astype(np.float32))

Re_in_tf      = tf.convert_to_tensor(Re_in_np.astype(np.float32))
Re_out_tf     = tf.convert_to_tensor(Re_out_np.astype(np.float32))
Re_top_tf     = tf.convert_to_tensor(Re_top_np.astype(np.float32))
Re_bot1_tf    = tf.convert_to_tensor(Re_bot1_np.astype(np.float32))
Re_bot2_tf    = tf.convert_to_tensor(Re_bot2_np.astype(np.float32))
Re_step_tf    = tf.convert_to_tensor(Re_step_np.astype(np.float32))
Re_corner_tf  = tf.convert_to_tensor(Re_corner_np.astype(np.float32))

# Build model
re_mean_log = np.log(500).astype(np.float32)
model = PurePINN(
    num_hidden_layers=D_default,
    num_neurons_per_layer=N_default,
    include_pressure=True,
    x_mean=( -L_in + L_out ) / 2,
    x_scale=( L_out + L_in ) / 2,
    y_mean=H / 2,
    y_scale=H / 2,
    re_mean=re_mean_log,
    re_scale=1.0,
    use_log_re=True
)

# Dummy input (must be inside fluid domain)
_dummy_input = tf.convert_to_tensor(
    np.array([[0.0, (h + H)/2, 500]], dtype=np.float32)
)

_ = model(_dummy_input)

# Model info
print("Model built. Trainable params:",
      sum([np.prod(v.shape) for v in model.trainable_variables]))

In [ ]:
# Convert tensors to numpy
Xc = X_interior_tf.numpy()
inlet   = inlet_tf.numpy()
outlet  = outlet_tf.numpy()
top     = top_tf.numpy()
bottom1 = bottom1_tf.numpy()
bottom2 = bottom2_tf.numpy()
step    = step_tf.numpy()

# Optional corner
try:
    corner = corner_tf.numpy()
except:
    corner = None

# Plot
plt.figure(figsize=(15, 3))

# Interior points
plt.scatter(Xc[:,0], Xc[:,1],
            s=1, color='lightgray', label='Collocation (interior)')

# Boundary points

# Inlet
plt.scatter(inlet[:,0], inlet[:,1],
            s=8, color='red', label='Inlet (velocity BC)')

# Outlet
plt.scatter(outlet[:,0], outlet[:,1],
            s=8, color='blue', label='Outlet (pressure BC)')

# Top wall
plt.scatter(top[:,0], top[:,1],
            s=8, color='green', label='Top wall (no-slip)')

# Bottom BEFORE step
plt.scatter(bottom1[:,0], bottom1[:,1],
            s=8, color='orange', label='Bottom wall (before step)')

# Bottom AFTER step
plt.scatter(bottom2[:,0], bottom2[:,1],
            s=8, color='gold', label='Bottom wall (after step)')

# Step wall
plt.scatter(step[:,0], step[:,1],
            s=8, color='purple', label='Step wall (no-slip)')

# Corner
if corner is not None:
    plt.scatter(corner[:,0], corner[:,1],
                s=20, color='black', label='Corner')

# Draw solid region (step block)
plt.fill_between([-L_in, 0], 0, h, color='white', alpha=0.5)

# Labels & styling
plt.xlabel("x")
plt.ylabel("y")
plt.title("Domain Sampling & Boundary Conditions (Backward-Facing Step)")

plt.legend(loc='upper right', fontsize=8)
# plt.axis("equal")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Adam optimization schedule
initial_lr = 5e-3
boundaries = [1000, 5000, 15000, 30000, 60000, 80000, 95000]
values = [5e-3, 1e-3, 5e-4, 1e-4, 5e-5, 1e-5, 5e-6, 1e-6]

lr_schedule = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=boundaries,
    values=values
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# Training settings
EPOCHS_ADAM = 100000
print_every = 1000
checkpoint_every = 1000

# History containers
loss_history = {
    "total": [],
    "Le1": [], "Le2": [], "Le3": [],
    "Lbc_v": [],
    "Lbc_p": []
}
grad_norm_history = []

# Epoch counter
epoch_var = tf.Variable(0, dtype=tf.int64)

# Checkpoint setup
checkpoint_dir = "./checkpoints_BFSF_PINN"
os.makedirs(checkpoint_dir, exist_ok=True)

ckpt = tf.train.Checkpoint(
    epoch=epoch_var,
    optimizer=optimizer,
    model=model
)

manager = tf.train.CheckpointManager(
    ckpt,
    checkpoint_dir,
    max_to_keep=3
)

# Restore
if manager.latest_checkpoint:
    ckpt.restore(manager.latest_checkpoint).expect_partial()
    print(f"\nRestored from checkpoint: {manager.latest_checkpoint}")
    print(f"Resuming from epoch {epoch_var.numpy()}")
else:
    print("\nStarting training from scratch")

start_epoch = int(epoch_var.numpy())

# GPU detection
gpus = tf.config.list_physical_devices('GPU')
num_gpus = len(gpus)

if num_gpus > 0:
    gpu_names = [gpu.name for gpu in gpus]
    print(f"\nDetected {num_gpus} GPU(s):")
    for i, gpu in enumerate(gpus):
        print(f"  GPU {i}: {gpu.name}")
else:
    gpu_names = ["CPU"]
    print("\nNo GPU detected. Training will run on CPU.")

# Training loop
start = time.time()

for epoch in range(start_epoch + 1, EPOCHS_ADAM + 1):

    loss_total, Le1, Le2, Le3, Lbc_v, Lbc_p, grad_norm = training_step(
        model, optimizer,
        X_interior_tf, Re_interior_tf,
        inlet_tf, outlet_tf, top_tf,
        bottom1_tf, bottom2_tf, step_tf, corner_tf,
        Re_in_tf, Re_out_tf, Re_top_tf,
        Re_bot1_tf, Re_bot2_tf, Re_step_tf, Re_corner_tf
    )

    epoch_var.assign(epoch)

    # Logging
    loss_history["total"].append(loss_total.numpy())
    loss_history["Le1"].append(Le1.numpy())
    loss_history["Le2"].append(Le2.numpy())
    loss_history["Le3"].append(Le3.numpy())
    loss_history["Lbc_v"].append(Lbc_v.numpy())
    loss_history["Lbc_p"].append(Lbc_p.numpy())
    grad_norm_history.append(grad_norm.numpy())

    # Print
    if epoch % print_every == 0 or epoch == 1:
        elapsed = time.time() - start
        print(
            f"[{epoch}/{EPOCHS_ADAM}] "
            f"loss={loss_total.numpy():.3e} "
            f"Le1={Le1.numpy():.3e} "
            f"Le2={Le2.numpy():.3e} "
            f"Le3={Le3.numpy():.3e} "
            f"Lbc_v={Lbc_v.numpy():.3e} "
            f"Lbc_p={Lbc_p.numpy():.3e} "
            f"GradNorm={grad_norm.numpy():.3e} "
            f"t={elapsed:.1f}s "
            f"GPUs: {num_gpus} ({', '.join(gpu_names)})"
        )

        start = time.time()

    # Checkpoint
    if epoch % checkpoint_every == 0:
        save_path = manager.save()
        print(f"Checkpoint saved at epoch {epoch}: {save_path}")

# Save history
np.save(os.path.join(checkpoint_dir, "loss_history.npy"), loss_history)
np.save(os.path.join(checkpoint_dir, "grad_norm_history.npy"), grad_norm_history)
print("\nTraining complete. Histories saved.")

In [ ]:
# L-BFGS Refinement Stage
lbfgs_history = {
    "total": [],
    "Le1": [],
    "Le2": [],
    "Le3": [],
    "Lbc_v": [],
    "Lbc_p": []
}

vars = model.trainable_variables
sizes = [int(tf.size(v)) for v in vars]

def pack_weights():
    return np.concatenate(
        [tf.reshape(v, [-1]).numpy() for v in vars]
    ).astype(np.float64)

def unpack_and_assign(vec):
    pos = 0
    for v, s in zip(vars, sizes):
        slice_val = vec[pos:pos+s].astype(np.float32)
        v.assign(tf.reshape(slice_val, v.shape))
        pos += s

def loss_and_grad(x0):

    unpack_and_assign(x0)
    with tf.GradientTape() as tape:

        # PDE
        r1, r2, r3 = pde_residuals(model, X_interior_tf, Re_interior_tf)
        Le1 = tf.reduce_mean(tf.square(r1))
        Le2 = tf.reduce_mean(tf.square(r2))
        Le3 = tf.reduce_mean(tf.square(r3))

        # BC
        Lbc_v, Lbc_p = boundary_losses(
            model,
            inlet_tf, outlet_tf, top_tf,
            bottom1_tf, bottom2_tf, step_tf, corner_tf,
            Re_in_tf, Re_out_tf, Re_top_tf,
            Re_bot1_tf, Re_bot2_tf, Re_step_tf, Re_corner_tf
        )

        # Weights
        lambda_v = 1.0
        lambda_p = 0.1
        Loss_total = Le1 + Le2 + Le3 + lambda_v * Lbc_v + lambda_p * Lbc_p

    # Store history
    lbfgs_history["total"].append(Loss_total.numpy())
    lbfgs_history["Le1"].append(Le1.numpy())
    lbfgs_history["Le2"].append(Le2.numpy())
    lbfgs_history["Le3"].append(Le3.numpy())
    lbfgs_history["Lbc_v"].append(Lbc_v.numpy())
    lbfgs_history["Lbc_p"].append(Lbc_p.numpy())

    # Gradients
    grads = tape.gradient(Loss_total, model.trainable_variables)
    grads = [tf.zeros_like(v) if g is None else g
             for g, v in zip(grads, model.trainable_variables)]

    grad_flat = tf.concat([tf.reshape(g, [-1]) for g in grads], axis=0)

    return float(Loss_total.numpy()), grad_flat.numpy().astype(np.float64)

# SciPy L-BFGS
x0 = pack_weights()
print(" Starting L-BFGS-B optimization...")
t0 = time.time()
result = spo.minimize(
    fun=loss_and_grad,
    x0=x0,
    method='L-BFGS-B',
    jac=True,
    options={
        'maxiter': 1000,
        'ftol': 1e-9,
        'maxcor': 50,
        'disp': True
    }
)

t1 = time.time()
print(f"\n L-BFGS finished: success={result.success}, "
      f"message='{result.message}', time={t1 - t0:.1f}s")

# Assign optimized weights
unpack_and_assign(result.x)
print(" Model weights updated with L-BFGS result.")

In [ ]:
combined_history = {}
for key in ["total", "Le1", "Le2", "Le3", "Lbc_v", "Lbc_p"]:
    adam_part = loss_history[key]
    lbfgs_part = lbfgs_history[key]
    combined_history[key] = np.concatenate([adam_part, lbfgs_part])

In [ ]:
# Plot loss curves
plt.figure(figsize=(8, 6))
plt.semilogy(loss_history["total"], label="Total Loss", linewidth=1)
plt.semilogy(loss_history["Le1"], label="Continuity Loss")
plt.semilogy(loss_history["Le2"], label="Momentum-x Loss")
plt.semilogy(loss_history["Le3"], label="Momentum-y Loss")
plt.semilogy(loss_history["Lbc_v"], label="BC Velocity Loss")
plt.semilogy(loss_history["Lbc_p"], label="BC Pressure Loss")
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss (log scale)", fontsize=12)
plt.title("Training Loss Evolution (Backward-Facing Step)", fontsize=14)
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Save losses
np.savez("loss_history.npz", **loss_history)
print("Saved loss history to 'loss_history.npz'")

In [ ]:
N_adam = len(loss_history["total"])
N_total = len(combined_history["total"])
x_adam = np.arange(N_adam)
x_lbfgs = np.arange(len(lbfgs_history["total"])) + N_adam
plt.figure(figsize=(10,7))

# Total loss
plt.semilogy(combined_history["total"], label="Total Loss", linewidth=2)

# PDE losses
plt.semilogy(combined_history["Le1"], '--', label="Continuity")
plt.semilogy(combined_history["Le2"], '--', label="Momentum-x")
plt.semilogy(combined_history["Le3"], '--', label="Momentum-y")

# BC losses
plt.semilogy(combined_history["Lbc_v"], ':', label="BC Velocity")
plt.semilogy(combined_history["Lbc_p"], ':', label="BC Pressure")

# Optimizer switch line
plt.axvline(x=N_adam, color='k', linestyle='--', linewidth=2, label='Adam → L-BFGS')

plt.xlabel("Iteration")
plt.ylabel("Loss (log scale)")
plt.title("Full Training Loss Evolution (Adam + L-BFGS)")

plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
print("Final Adam loss:", loss_history["total"][-1])
print("Final L-BFGS loss:", lbfgs_history["total"][-1])

In [ ]:
# Save trained PINN model (.keras format)
model.save("BFSReDataFreePINNs.keras")

In [ ]:

# Grid for plotting results after trainning the model
Nx, Ny = 200, 120
x = np.linspace(x_min, x_max, Nx)
y = np.linspace(y_min, y_max, Ny)
X, Y = np.meshgrid(x, y)

# Mask solid region (step block)
mask = (X < 0.0) & (Y > h)

# Reynolds number input
Re_plot = 1e2
Re_val = np.full_like(X, Re_plot)

# Flatten
X_flat = X.flatten()
Y_flat = Y.flatten()
Re_flat = Re_val.flatten()

# Keep only fluid points
valid = ~( (X_flat < 0.0) & (Y_flat < h) )

X_input = np.stack([
    X_flat[valid],
    Y_flat[valid],
    Re_flat[valid]
], axis=1)

X_tf = tf.convert_to_tensor(X_input, dtype=tf.float32)

# Model prediction
u_pred, v_pred, p_pred = model(X_tf)
u_pred = u_pred.numpy().flatten()
v_pred = v_pred.numpy().flatten()
p_pred = p_pred.numpy().flatten()

# Rebuild full fields with NaNs
U = np.full_like(X_flat, np.nan, dtype=np.float64)
V = np.full_like(X_flat, np.nan, dtype=np.float64)
P = np.full_like(X_flat, np.nan, dtype=np.float64)

U[valid] = u_pred
V[valid] = v_pred
P[valid] = p_pred

U = U.reshape(Ny, Nx)
V = V.reshape(Ny, Nx)
P = P.reshape(Ny, Nx)

vel_mag = np.sqrt(U**2 + V**2)

# Function for plotting
def plot_field(field, title, label):
    plt.figure(figsize=(10, 2))
    contour = plt.contourf(X, Y, field, levels=200, cmap='jet')
    plt.colorbar(contour, label=label)

    # Draw step geometry
    plt.fill_between([-L_in, 0], 0, h, color='white')
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.show()

# Plots
plot_field(vel_mag, f"Velocity Magnitude (Re={Re_plot})", "|u|")
plot_field(U, f"u-velocity (Re={Re_plot})", "u")
plot_field(V, f"v-velocity (Re={Re_plot})", "v")
plot_field(P, f"Pressure (Re={Re_plot})", "p")

In [ ]:
# Streamline plot
plt.figure(figsize=(10,2))
# Background contour
contour = plt.contourf(X, Y, vel_mag, levels=200, cmap='jet')
plt.colorbar(contour, label="|u|")

# Streamlines on top
plt.streamplot(
    X, Y,
    np.ma.masked_invalid(U),
    np.ma.masked_invalid(V),
    color='k',
    density=2,
    linewidth=0.7
)

# Step
plt.fill_between([-L_in, 0], 0, h, color='white')
plt.title(f"Velocity + Streamlines (Re = {Re_plot})")
plt.xlabel("x")
plt.ylabel("y")
plt.show()